In [0]:
from pyspark.sql.functions import col, monotonically_increasing_id, current_timestamp, current_date, lit, when, floor, datediff, coalesce
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# =============================================================
# CONFIGURATION
# =============================================================
print("="*80)
print("CREATING DIM_CUSTOMER (SCD TYPE 2) FROM SILVER LAYER")
print("="*80)

silver_customers_path = "s3://travel-analytics-bronze/delta/silver/customer/"
gold_dim_customer_path = "s3://travel-analytics-bronze/delta/gold/Dim_Customer/"

# =============================================================
# STEP 1: LOAD SILVER DATA (Already Transformed)
# =============================================================
print("\nSTEP 1: Loading Silver Customers Data...")

silver_df = spark.read.format("delta").load(silver_customers_path)

print(f"Loaded {silver_df.count():,} customer records from silver")

# =============================================================
# STEP 2: ADD DERIVED COLUMNS & SCD TYPE 2 METADATA
# =============================================================
print("\nSTEP 2: Creating Dimension Structure with SCD Type 2...")

# Add surrogate key using window function for better control
window_spec = Window.orderBy("Customer_Id")

dim_customer_df = (
    silver_df
    
    # Add surrogate key
    .withColumn("Dim_Customer_SK", monotonically_increasing_id() + 1)
    
    # Add SCD Type 2 columns
    .withColumn("Valid_From", 
        coalesce(col("Updated_At").cast("date"), current_date())
    )
    .withColumn("Valid_To", lit(None).cast("date"))
    .withColumn("Is_Current", lit(True))
    
    # Select and rename columns to match dimension schema
    .select(
        col("Dim_Customer_SK"),                        # PK - Surrogate Key
        col("Customer_Id").alias("Customer_ID_BK"),    # Business Key
        col("First_Name"),
        col("Family_Name"),
        col("Birth_Date").alias("Date_Of_Birth"),
        col("Gender"),
        col("Phone_Number").alias("Phone"),
        col("Email"),
        col("Country"),
        col("Valid_From"),
        col("Valid_To"),
        col("Is_Current")
    )
)



CREATING DIM_CUSTOMER (SCD TYPE 2) FROM SILVER LAYER

STEP 1: Loading Silver Customers Data...
Loaded 1,000 customer records from silver

STEP 2: Creating Dimension Structure with SCD Type 2...


In [0]:
dim_customer_df.display()

Dim_Customer_SK,Customer_ID_BK,First_Name,Family_Name,Date_Of_Birth,Gender,Phone,Email,Country,Valid_From,Valid_To,Is_Current
1,5812371,Leonce,Tignon,1970-05-06,F,903515307491,leonce.tignon.5812371@company.com,Turkey,2025-12-12,null,true
2,8826120,Candice,Rabouin,1979-11-09,F,4761189426,candice.rabouin@example.com,Norway,2025-12-12,null,true
3,5710465,Rodolphe,Sourice,1979-06-25,M,490258983885,rodolphe.sourice@example.com,Germany,2025-12-12,null,true
4,7848734,Francois,Bellard,1987-05-13,M,21691212698,francois.bellard.7848734@company.com,Tunisia,2025-12-12,null,true
5,3402948,Nadege,Morand,1976-10-14,F,34210397203,nadege.morand@example.com,Spain,2025-12-12,null,true
6,6657801,Amaury,Rouleau,1977-12-27,M,918105127088,amaury.rouleau.6657801@company.com,India,2025-12-12,null,true
7,4685975,Marie-catherine,Royer,1981-10-29,F,76283937917,mariecatherine.royer.4685975@company.com,Russia,2025-12-12,null,true
8,7905113,Rosemarie,Nourry,1980-06-15,F,71890323794,rosemarie.nourry.7905113@company.com,Russia,2025-12-12,null,true
9,2016379,Helga,Chailloux,1982-12-06,F,78168571640,helga.chailloux.2016379@company.com,Russia,2025-12-12,null,true
10,7282871,Ericka,Douillard,1975-10-04,F,212529153954,ericka.douillard.7282871@company.com,Morocco,2025-12-12,null,true


In [0]:
# =============================================================
# STEP 3: WRITE TO GOLD LAYER (INITIAL LOAD)
# =============================================================
print("\nSTEP 3: Writing to Gold Layer...")

dim_customer_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_dim_customer_path)

print(f" Saved to: {gold_dim_customer_path}")


STEP 3: Writing to Gold Layer...
 Saved to: s3://travel-analytics-bronze/delta/gold/Dim_Customer/


In [0]:

# =============================================================
# STEP 4: CREATE MANAGED TABLE
# =============================================================
print("\nSTEP 4: Creating Managed Table...")

# Create gold schema if not exists
spark.sql("CREATE SCHEMA IF NOT EXISTS awsdata.gold")

# Create table
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS awsdata.gold.Dim_Customer (
        Dim_Customer_SK BIGINT COMMENT 'Surrogate key',
        Customer_ID_BK INTEGER COMMENT 'Business key from source system',
        First_Name STRING,
        Family_Name STRING,
        Date_Of_Birth DATE,
        Gender STRING,
        Phone STRING,
        Email STRING,
        Country STRING,
        City STRING,
        age_category STRING COMMENT 'Derived: Under 18, 18-30, 31-50, Over 50',
        Valid_From DATE COMMENT 'SCD Type 2: Start date of validity',
        Valid_To DATE COMMENT 'SCD Type 2: End date of validity (NULL for current)',
        Is_Current BOOLEAN COMMENT 'SCD Type 2: Flag for current record'
    )
    USING DELTA
    LOCATION '{gold_dim_customer_path}'
    COMMENT 'Customer dimension - SCD Type 2'
""")

print("    Table created: awsdata.gold.Dim_Customer")

# =============================================================
# STEP 5: OPTIMIZE & ADD INDEXES
# =============================================================
print("\nSTEP 5: Optimizing...")

spark.sql("OPTIMIZE awsdata.gold.Dim_Customer")
spark.sql("ANALYZE TABLE awsdata.gold.Dim_Customer COMPUTE STATISTICS")

# Add Z-ordering for better query performance on common filters
spark.sql("OPTIMIZE awsdata.gold.Dim_Customer ZORDER BY (Customer_ID_BK, Is_Current)")

print("    Optimization complete")

# =============================================================
# STEP 6: VALIDATION
# =============================================================
print("\n" + "="*80)
print("VALIDATION")
print("="*80)

result_df = spark.table("awsdata.gold.Dim_Customer")

print(f"\n Dim_Customer created successfully!")
print(f"   Records: {result_df.count():,}")
print(f"   Columns: {len(result_df.columns)}")
print(f"   Location: {gold_dim_customer_path}")

# Check SCD Type 2 setup
current_records = result_df.filter(col("Is_Current") == True).count()
print(f"   Current records: {current_records:,}")

print("\n📄 Sample:")
display(result_df.orderBy("Customer_ID_BK").limit(10))

print("\n📊 Age Category Distribution:")
display(
    result_df.filter(col("Is_Current") == True)
    .groupBy("age_category")
    .count()
    .orderBy("age_category")
)

print("\n" + "="*80)
print("✅ COMPLETE!")
print("="*80)